# Computational Model of Message Sensation Value (MSV)

In this notebook, you will be able to calculate the MSV of a short video from uploading the video to multimodal feature extraction and MSV estimation.

This notebook supplements the manuscript: **A Computational Model of Message Sensation Value in Short Video Multimodal Features that Predicts Sensory and Behavioral Engagement** by Haoning Xue, Jingwen Zhang, Xiaohui Wang, Diane Dagyong Kim, and Yunya Song.

If you have any questions, feel free to contact **Haoning Xue** via haoning.xue@utah.edu.

## Set-up

This section includes basic library import and Application Programming Interface (API) setting.

In [ ]:
%pip install --upgrade google-cloud-videointelligence
%pip install python-facepp
%pip install fer opencv-python tensorflow
%pip install librosa

In [ ]:
import cv2
from fer.fer import FER
from fer.classes import Video
from moviepy.editor import VideoFileClip
import librosa
import pandas as pd
import numpy as np
import datetime
import re, csv
import webcolors
import math, statistics
import os, sys, glob, io, base64
import tempfile
import requests
import pickle
import json
from time import sleep
from google.cloud import videointelligence_v1 as videointelligence
from facepplib import FacePP, exceptions
import joblib
from sklearn.preprocessing import StandardScaler
np.set_printoptions(suppress=True, precision=6)

In [ ]:
""" API Setup: Face++ """

# go to https://www.faceplusplus.com/ and set up an API
url = "https://api-us.faceplusplus.com/facepp/v3/detect"
api_key = "your-api-key"
api_secret = "your-api-secret"

In [ ]:
""" API Setup: Google Cloud Video Intelligence API """

# go to https://cloud.google.com/video-intelligence and set up API
cloud_key = "your-google-cloud-key" # upload credentials json file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = cloud_key

video_client = videointelligence.VideoIntelligenceServiceClient()
video_client.from_service_account_json(cloud_key)

# videointelligence.Feature.SHOT_CHANGE_DETECTION
shot_feature = [videointelligence.Feature.SHOT_CHANGE_DETECTION]

# videointelligence.Feature.FACE_DETECTION
face_feature = [videointelligence.Feature.FACE_DETECTION]
face_config = videointelligence.types.FaceDetectionConfig(include_bounding_boxes=True, include_attributes=True)
face_context = videointelligence.types.VideoContext(face_detection_config=face_config)

In [ ]:
""" Import video """
project_folder = "your-folder-path"
temp_video_path = "test-video.mp4"

## Visual Feature Extraction

This section includes visual feature extraction: (1) brightness, (2) saturation, (3) warmness, (4) visual complexity, (5) shot count per second, and (6) shot duration per second. The first four features are frame-level features and one frame is sampled in every 30 frames in this example.

In [ ]:
"""Frame-level feature functions"""

def get_brightness(frame):
  """color brightness"""
  lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
  bright = lab[...,0].mean()
  return bright

def get_saturation(frame):
  """average color saturation"""
  frame_hls = cv2.cvtColor(frame, cv2.COLOR_BGR2HLS)
  sat = frame_hls[:, :, 2]
  avg_sat = round(np.mean(sat),2)
  return avg_sat

def get_entropy(frame):
  """visual complexity"""
  gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
  signal = gray[0:gray.shape[0],0:gray.shape[1]].flatten()
  lensig=signal.size
  symset=list(set(signal))
  numsym=len(symset)
  propab=[np.size(signal[signal==i])/(1.0*lensig) for i in symset]
  ent=np.sum([p*np.log2(1.0/p) for p in propab])
  return ent

def get_warmness_ratio(frame):
  """get warmness ratio of a frame"""
  frame_hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
  lower_warm = np.array([0, 50, 50])
  upper_warm = np.array([30, 255, 255])
  mask = cv2.inRange(frame_hsv, lower_warm, upper_warm)
  num_warm_pixels = np.count_nonzero(mask)
  total_pixels = frame.shape[0] * frame.shape[1]
  warmness_ratio = num_warm_pixels / total_pixels
  return warmness_ratio

In [ ]:
""" Frame-level visual feature extraction """
video_cap = cv2.VideoCapture(temp_video_path)
total_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))

# initiate feature lists
frame_id = []
brightness = []
saturation = []
entropy = []
warmness = []

# extract features for every 6 frames
for j in range(total_frames):
  if j%30 == 0:
    video_cap.set(cv2.CAP_PROP_POS_FRAMES, j)
    ret, frame = video_cap.read()
    if ret is True:
      frame_id.append(j)
      brightness.append(get_brightness(frame))
      saturation.append(get_saturation(frame))
      entropy.append(get_entropy(frame))
      warmness.append(get_warmness_ratio(frame))

brightness = np.array(brightness)
saturation = np.array(saturation)
entropy = np.array(entropy)
warmness = np.array(warmness)

In [ ]:
""" Shot count & shot duration per second extraction """

with open(temp_video_path, 'rb') as video_file:
  input_content = video_file.read()

operation = video_client.annotate_video(request={"features": shot_feature, "input_content": input_content})
result = operation.result()
annotation_result = result.annotation_results[0]

# initiate shot change feature list
shot_start_time = []
shot_end_time = []

for k, shot in enumerate(annotation_result.shot_annotations):
  start_time = (shot.start_time_offset.seconds + shot.start_time_offset.microseconds / 1e6)
  end_time = (shot.end_time_offset.seconds + shot.end_time_offset.microseconds / 1e6)
  shot_start_time.append(start_time)
  shot_end_time.append(end_time)

In [ ]:
print(f"#frames sampled: {len(frame_id)}\n"
      f"brightness: {brightness.mean():.2f}\n"
      f"saturation: {saturation.mean():.2f}\n"
      f"visual complexity: {entropy.mean():.2f}\n"
      f"warmness: {warmness.mean():.2f}\n"
      f"shot count: {len(shot_start_time):.2f}\n"
      f"shot duration: {np.mean([x - y for x, y in zip(shot_end_time, shot_start_time)]):.2f}")

## Audio Feature Extraction

This section includes audio feature extraction: (1) loudness, (2) tempo, (3) sound brightness.

In [ ]:
temp_video = VideoFileClip(temp_video_path)
temp_audio = temp_video.audio

In [ ]:
if temp_audio is not None:
  temp_audio_path = project_folder + 'temp.wav'
  temp_audio.write_audiofile(temp_audio_path, logger=None)

  y, sr = librosa.load(temp_audio_path)
  tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
  rms = librosa.feature.rms(y=y)[0][0]
  spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]

print(f"tempo: {tempo[0]:.2f}\n"
      f"loudness: {rms:.2f}\n"
      f"sound brightness: {spec_centroid.mean():.2f}")

## Face Feature Extraction

This section includes face feature extraction: (1) the number of faces per second and (2) the duration of faces per second.

In [ ]:
""" Face count & face duration per second extraction """
operation = video_client.annotate_video(request={"features": face_feature, "input_content": input_content, "video_context": face_context})
result = operation.result(timeout = 10000)
annotation_result = result.annotation_results[0]

# initiate feature lists
face_start_time = []
face_end_time = []

for annotation in annotation_result.face_detection_annotations:
  for track in annotation.tracks:
    face_start_time.append(track.segment.start_time_offset.seconds + track.segment.start_time_offset.microseconds / 1e6)
    face_end_time.append(track.segment.end_time_offset.seconds + track.segment.end_time_offset.microseconds / 1e6)

In [ ]:
print(f"face count: {len(face_start_time):.2f}\n"
      f"face duration: {np.mean([x - y for x, y in zip(face_end_time, face_start_time)]):.2f}\n")

## MSV Calculation

This section presents the calculation of MSV based on 11 multimodal features: tempo, loudness (i.e., rms), sound brightness (i.e., spectral centroid), brightness, saturation, visual complexity (i.e., entropy), warmness, face count per second, shot count per second, face duration per second, shot duration per second.

In [ ]:
# data preparation
video_duration = temp_video.duration
face_count_ps = len(face_start_time)/video_duration
shot_count_ps = len(shot_start_time)/video_duration
face_duration_ps = np.mean([x - y for x, y in zip(face_end_time, face_start_time)])/video_duration
shot_duration_ps = np.mean([x - y for x, y in zip(shot_end_time, shot_start_time)])/video_duration

# dataframe preparation
feature_11 = [tempo[0], rms, spec_centroid.mean(), brightness.mean(), saturation.mean(),
              entropy.mean(), warmness.mean(), face_count_ps, shot_count_ps, face_duration_ps, shot_duration_ps]
feature_cols = ['tempo', 'loudness', 'spec_centroid', 'brightness', 'saturation',
                'entropy', 'warmness', 'facec_ps', 'shotc_ps', 'faced_ps', 'shotd_ps']
feature_df = pd.DataFrame([feature_11], columns=feature_cols)

# load the saved MSV scaler & model
scaler = joblib.load('../model/msv_scaler.save')
rf_model = joblib.load('../model/msv_model.joblib')

# data standardization
feature_df[feature_cols] = scaler.transform(feature_df[feature_cols])

# MSV prediction
rf_model.predict(feature_df[feature_cols])